## Get Physical Properties from AlphaFold Output

May need `openmmtools` and `openff-toolkit` from `conda-forge` and `omnia`

In [ ]:
# Install packages
%pip install -q numpy pandas matplotlib seaborn pathos biopython tdqm mdtraj rdkit MDAnalysis prolif
%pip install -q openmm openmmtools openff-toolkit

: 

## Setup

In [1]:
import prolif as plf
import MDAnalysis as mda
from MDAnalysis.analysis import contacts
from rdkit import Chem
from rdkit.Chem import AllChem
import os
import numpy as np

C:\Users\ryangustafson\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\MDAnalysis\topology\tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)


In [ ]:
import os
from glob import glob
import numpy as np
import mdtraj as md
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from Bio.PDB import PDBParser, PDBIO, Select

from openmm import app, unit, Platform
import openmm as mm
from openmmtools import mbar, states, alchemy
from openff.toolkit.topology import Molecule

import prolif as plf

from pathos.multiprocessing import ProcessingPool as Pool


ModuleNotFoundError: No module named 'openff'

In [9]:
class ChainSelect(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id
    def accept_chain(self, chain):
        return chain.id == self.chain_id

def split_chains(pdb_file):
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("complex", pdb_file)

    ligand_file = pdb_file.replace(".pdb", "_ligand.pdb")
    protein_file = pdb_file.replace(".pdb", "_protein.pdb")

    io = PDBIO()
    
    # Save ligand (chain A)
    io.set_structure(struct)
    io.save(ligand_file, select=ChainSelect("A"))
    
    # Save protein (chain B)
    io.save(protein_file, select=ChainSelect("B"))

    return ligand_file, protein_file


NameError: name 'Select' is not defined

In [ ]:
def build_system(ligand_pdb, protein_pdb):
    # Load protein
    protein = app.PDBFile(protein_pdb)
    modeller = app.Modeller(protein.topology, protein.positions)

    # Parametrize ligand
    ligand = Molecule.from_file(ligand_pdb)
    ligand_ff = ligand.to_topology().to_openmm()
    ligand_positions = ligand.conformers[0]

    modeller.add(ligand_ff, ligand_positions)

    # Add solvent
    modeller.addHydrogens()
    modeller.addSolvent(app.ForceField('amber14-all.xml', 'amber14/tip3p.xml'),
                        model='tip3p',
                        padding=1.0*unit.nanometers)

    forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3p.xml')
    system = forcefield.createSystem(modeller.topology,
                                     nonbondedMethod=app.PME,
                                     nonbondedCutoff=1.0*unit.nanometers,
                                     constraints=app.HBonds)
    return modeller, system


In [ ]:
folder = "path_to_pdb_folder"
results = []

for pdb_file in tqdm(glob(folder + "/*.pdb")):
    ligand_pdb, protein_pdb = split_chains(pdb_file)

    modeller, system = build_system(ligand_pdb, protein_pdb)
    outprefix = pdb_file.replace(".pdb", "_md")

    traj = run_md(modeller, system, outprefix)

    analysis = analyze(protein_pdb, ligand_pdb, traj)
    analysis["complex"] = os.path.basename(pdb_file)

    results.append(analysis)

df = pd.DataFrame(results)
df.to_csv("md_results.csv", index=False)
df


In [ ]:
def analyze(protein_pdb, ligand_pdb, traj):
    traj_md = md.load(traj, top=protein_pdb)
    ligand_traj = md.load(traj, top=ligand_pdb)

    rmsd = md.rmsd(ligand_traj, ligand_traj, 0)
    hbonds = md.baker_hubbard(traj_md)

    return {
        "rmsd_mean": rmsd.mean(),
        "rmsd_max": rmsd.max(),
        "hbonds_count": len(hbonds)
    }


## MDAnalysis

In [7]:
import MDAnalysis as mda
from MDAnalysis.lib.distances import distance_array
import numpy as np

# Load your PDB file
pdb_file = "my_complex.pdb" # <-- Your PDB file here

try:
    u = mda.Universe(pdb_file)
except Exception as e:
    print(f"Error loading {pdb_file}: {e}")
    exit()

# --- 1. Define Your Specific Parts ---
sel_A = "chainID A"
sel_B = "chainID B"

group_A = u.select_atoms(sel_A)
group_B = u.select_atoms(sel_B)

if group_A.n_atoms == 0 or group_B.n_atoms == 0:
    print("Error: One or both selections resulted in 0 atoms.")
    print(f"Check if '{sel_A}' and '{sel_B}' are correct for your PDB.")
    exit()

print(f"Group A ({sel_A}): {group_A.n_atoms} atoms")
print(f"Group B ({sel_B}): {group_B.n_atoms} atoms")

# --- 2. Run Analysis (New Method) ---
distance_cutoff = 4.5

# Calculate the distance matrix between all atoms in group A
# and all atoms in group B.
# This creates a (n_atoms_A, n_atoms_B) array.
dist_matrix = distance_array(group_A.positions, group_B.positions)

# Find the indices (i, j) where the distance is less than the cutoff
# i = index in group_A, j = index in group_B
close_atom_indices = np.where(dist_matrix <= distance_cutoff)

# 'close_atom_indices' is a tuple of two arrays:
# (array_of_i_indices, array_of_j_indices)

# --- 3. Map Atom Indices to Residues ---

# Get the indices of the atoms in group_A that are close
close_atoms_A_indices = close_atom_indices[0]
# Get the indices of the atoms in group_B that are close
close_atoms_B_indices = close_atom_indices[1]

# Now, map these atom indices to their parent residues
# We use a set to store the residues so we only get unique ones
interacting_residues_A = set()
for atom_index in close_atoms_A_indices:
    interacting_residues_A.add(group_A[atom_index].residue)

interacting_residues_B = set()
for atom_index in close_atoms_B_indices:
    interacting_residues_B.add(group_B[atom_index].residue)

# --- 4. Print Results ---
print("\n--- MDAnalysis Contact Analysis (within 4.5 Å) ---")

print(f"\nResidues in Chain A interacting with Chain B:")
if interacting_residues_A:
    # Sort the residues by resid for a clean output
    sorted_residues = sorted(list(interacting_residues_A), key=lambda r: r.resid)
    print([res.resname + str(res.resid) for res in sorted_residues])
else:
    print("None found.")

print(f"\nResidues in Chain B interacting with Chain A:")
if interacting_residues_B:
    sorted_residues = sorted(list(interacting_residues_B), key=lambda r: r.resid)
    print([res.resname + str(res.resid) for res in sorted_residues])
else:
    print("None found.")

# --- Optional: Hydrogen Bond Analysis ---
# (This still has the same limitation: it needs hydrogens in the PDB)
try:
    from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
    
    # Select all protein atoms for H-bond analysis
    # We select 'protein' and 'protein' to find intra-protein H-bonds
    h = HydrogenBondAnalysis(u, sel1="protein", sel2="protein")
    h.run()
    
    # Filter for H-bonds specifically between Chain A and Chain B
    inter_chain_hbonds = []
    for bond in h.results.hbonds:
        donor_chain = u.atoms[bond[2]].chainID
        acceptor_chain = u.atoms[bond[0]].chainID
        
        if (donor_chain == 'A' and acceptor_chain == 'B') or \
           (donor_chain == 'B' and acceptor_chain == 'A'):
            inter_chain_hbonds.append(bond)

    print(f"\nFound {len(inter_chain_hbonds)} total inter-chain H-bonds (A-B).")
    # Note: This will likely be 0 if your PDB has no hydrogens.

except ImportError:
    print("\nHydrogen bond analysis requires an updated version of MDAnalysis.")
except Exception as e:
    print(f"\nCould not run H-bond analysis: {e}")

Group A (chainID A): 1520 atoms
Group B (chainID B): 1299 atoms

--- MDAnalysis Contact Analysis (within 4.5 Å) ---

Residues in Chain A interacting with Chain B:
['CYS1', 'THR2', 'CYS3', 'SER4', 'PRO5', 'PRO33', 'PHE34', 'GLY35', 'GLU62', 'SER64', 'GLU65', 'SER66', 'LEU67', 'CYS68', 'LYS71', 'ARG84', 'LEU94', 'CYS95', 'LYS123', 'LYS125', 'TYR128', 'TYR129', 'ASN148']

Residues in Chain B interacting with Chain A:
['PHE4', 'TYR73', 'ASP76', 'ASP79', 'GLY80', 'LEU81', 'LEU82', 'ALA83', 'HIS84', 'ALA85', 'PHE86', 'PRO87', 'ILE92', 'GLN93', 'GLY107', 'LYS108', 'GLN110', 'TYR112', 'VAL117', 'HIS120', 'GLU121', 'HIS124', 'ASP129', 'HIS130', 'PRO140', 'MET141', 'TYR142', 'ARG143', 'PHE144', 'GLU146']

Could not run H-bond analysis: HydrogenBondAnalysis.__init__() got an unexpected keyword argument 'sel1'. Did you mean 'self'?


In [ ]:
# %%
"""
OpenMM protein-ligand pipeline (Jupytext-style Python file you can open as a notebook)

Filename: OpenMM_protein_ligand_pipeline.py

What this does:
- Loops over a folder of PDB files (chain A = ligand, chain B = protein)
- Cleans PDBs with PDBFixer, adds hydrogens
- Parameterizes ligand using OpenFF (recommended) or a placeholder to use AmberTools
- Builds an OpenMM System with an AMBER protein force field and the ligand parameters
- Minimizes, equilibrates, and runs a short production MD on GPU
- Saves trajectory (.dcd) and topology (.pdb) per complex
- Performs basic analyses: RMSD, ligand COM distance, hydrogen bonds (via MDAnalysis)
- Exports snapshots for MM/PBSA with AmberTools (if you have MMPBSA.py)

Notes:
- This file is written as a jupytext-style script. To use it as a notebook: either
  - Install jupytext and open this file directly in Jupyter, or
  - Save as .py and run cells in an editor that supports # %% cells, or
  - Copy/paste into a new notebook.

Requirements (high level):
- Python 3.8+
- openmm, openmmforcefields, openff-toolkit, pdbfixer, rdkit, mdtraj or MDAnalysis, numpy, pandas, matplotlib
- Optional but recommended: ambertools (for MM/PBSA), conda-forge channel recommended for many packages

Recommended quick conda env (example):
  conda create -n omm-md python=3.10 -c conda-forge openmm openmmforcefields openff-toolkit pdbfixer rdkit mdtraj mdanalysis numpy pandas matplotlib jupyterlab
  # If you use ambertools for MM/PBSA:
  conda install -c conda-forge ambertools

Run length choices: for demo use short runs (e.g., 2 ns). For meaningful results use >= 20 ns and replicates.

"""

# %%
# --- User configuration (edit these) ---
INPUT_DIR = "./pdb_inputs"        # directory with PDBs
OUTPUT_DIR = "./outputs"         # where results will be written (per-pdb subfolders)
PH = 7.0
BOX_PADDING = 10.0                 # angstroms padding around complex for solvation box
PLATFORM_NAME = "CUDA"           # 'CUDA' or 'OpenCL' or 'CPU'
GPU_DEVICE_INDEX = "0"           # if you have multiple GPUs
N_STEPS_MINIMIZE = 5000
N_STEPS_EQ_NVT = 50000            # ~100 ps with 2 fs timestep
N_STEPS_EQ_NPT = 100000           # ~200 ps
N_STEPS_PRODUCTION = 500000       # with 2 fs this is 1 ns; change as needed
SAVE_EVERY_N_STEPS = 2500         # save frames every this many steps (~5 ps if dt=2 fs)
TEMPERATURE = 300.0
PRESSURE = 1.0
DT = 0.002                         # 2 fs
CONSTRAIN_HBONDS = True
PLATFORM_PROPERTIES = {"DeviceIndex": GPU_DEVICE_INDEX}

# %%
# Install note cell (for notebook users):
# If you need to install packages from within the notebook, uncomment and run the next cells.
# (In many setups it's better to create a conda env before launching Jupyter.)

# %%
# !pip install openmm openmmforcefields openff-toolkit pdbfixer rdkit-pypi mdtraj mdanalysis numpy pandas matplotlib jupyterlab
# !conda install -c conda-forge ambertools -y  # optional for MM/PBSA

# %%
# Imports
import os
import sys
import shutil
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Molecular libraries
from pdbfixer import PDBFixer
from openmm import app
import openmm as mm
from openmm import unit
from openmm.app import PDBFile, Modeller

# OpenFF toolkit for ligand parameterization
from openff.toolkit.topology import Molecule
from openff.toolkit.typing.engines.smirnoff import ForceField as OFFForceField
from openff.toolkit.utils import RDKitToolkitWrapper

# openmmforcefields to get common protein forcefields
from openmmforcefields.generators import SystemGenerator

# Trajectory/analysis
import mdtraj as md
import MDAnalysis as mda
from MDAnalysis.analysis import rms, hbonds

# %%
# Utility functions

def make_output_dirs(base_out: str, pdb_name: str):
    out = Path(base_out) / pdb_name
    out.mkdir(parents=True, exist_ok=True)
    (out / "analysis").mkdir(exist_ok=True)
    (out / "snapshots").mkdir(exist_ok=True)
    return out


def load_and_fix_pdb(pdb_path: str, keep_chains=("A", "B"), ph=7.0):
    """Load a PDB, keep only specified chains, run PDBFixer to add hydrogens and fix missing atoms.
    Returns a PDBFile and a Modeller object with hydrogens added.
    """
    pdb_path = str(pdb_path)
    fixer = PDBFixer(filename=pdb_path)
    # remove all chains not in keep_chains
    chains_to_keep = [c for c in fixer.topology.chains() if c.id in keep_chains]
    # PDBFixer doesn't have an easy chain removal API, so simplest approach: re-save only wanted atoms
    # We'll create a temporary PDB with just selected chains
    temp_pdb = tempfile.NamedTemporaryFile(suffix=".pdb", delete=False)
    with open(pdb_path) as fh, open(temp_pdb.name, "w") as out:
        write = False
        for line in fh:
            if line.startswith("ATOM") or line.startswith("HETATM") or line.startswith("TER") or line.startswith("MODEL") or line.startswith("ENDMDL"):
                chain_id = line[21].strip()
                if chain_id in keep_chains:
                    out.write(line)
            else:
                # copy header lines if needed
                out.write(line)
    fixer = PDBFixer(filename=temp_pdb.name)
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(pH=ph)
    # Save fixed PDB to a Modeller
    pdbfile = PDBFile(temp_pdb.name)
    modeller = Modeller(pdbfile.topology, pdbfile.positions)
    # modeller already has hydrogens via fixer; but we'll re-add to ensure
    return pdbfile, modeller


# %%
# Ligand extraction + OpenFF parameterization

def extract_ligand_to_sdf(modeller: Modeller, ligand_chain="A", out_sdf_path="ligand.sdf"):
    """Extract ligand atoms (chain A) from modeller.topology and save to SDF using RDKit via OpenMM positions.
    Returns the RDKit molecule via OpenFF or None on failure.
    """
    # Create a PDB file with only the ligand atoms
    ligand_atoms = [a for a in modeller.topology.atoms() if a.residue.chain.id == ligand_chain]
    if len(ligand_atoms) == 0:
        raise RuntimeError("No ligand atoms found in chain {}".format(ligand_chain))
    # Make a new topology/positions only with ligand
    ligand_top = app.Topology()
    chain = ligand_top.addChain(ligand_chain)
    res_map = {}
    for res in {a.residue for a in ligand_atoms}:
        new_res = ligand_top.addResidue(res.name, chain)
        for atom in res.atoms():
            ligand_top.addAtom(atom.name, atom.element, new_res)
    # positions: select by index order
    ligand_positions = []
    for atom in modeller.topology.atoms():
        if atom.residue.chain.id == ligand_chain:
            ligand_positions.append(modeller.positions[atom.index])
    # Save to temporary PDB and then use OpenFF to read
    tmp = tempfile.NamedTemporaryFile(suffix=".pdb", delete=False)
    with open(tmp.name, "w") as fh:
        PDBFile.writeFile(ligand_top, ligand_positions, fh)
    # Use OpenFF Molecule.from_file (which will use RDKit)
    off_mol = Molecule.from_file(tmp.name, toolkit_registry=RDKitToolkitWrapper())
    # Save as SDF
    off_mol.to_file(out_sdf_path, file_format="sdf")
    return off_mol


def parameterize_ligand_openff(off_mol: Molecule, off_forcefield_name="openff-2.0.0.offxml"):
    """Parameterize a ligand molecule using OpenFF and return an OpenMM Topology/System-like object for the ligand.
    We will create an OpenMM-compatible Topology/System by creating an OpenFF Topology and then creating an OpenMM System via the OFF ForceField.
    """
    # Create an OpenFF Topology from molecule
    topology = off_mol.to_topology()
    ff = OFFForceField(off_forcefield_name)
    # create an OpenMM system for the ligand in vacuum
    openff_topology = topology
    # We need an OpenMM-compatible topology object -> use off_mol.to_topology().to_openmm()
    omm_top = openff_topology.to_openmm()
    # create system
    # Use the OpenFF forcefield to create parameters
    system = ff.create_openmm_system(openff_topology)
    return system, omm_top


# %%
# Build whole system with protein + ligand using openmmforcefields SystemGenerator

def build_system(modeller: Modeller, ligand_system, ligand_omm_top, box_padding=BOX_PADDING):
    """Build a solvated OpenMM System combining protein and ligand params.
    ligand_system: OpenMM System for ligand (from OpenFF)
    ligand_omm_top: OpenMM Topology for ligand
    Returns: system, topology, positions
    """
    # First: create a combined topology by creating PDBFile of modeller and then loading it with app.Modeller
    # We'll use openmmforcefields SystemGenerator to create force field for protein and small molecules
    # Create an explicit system generator with protein and small molecule handlers
    ffxmls = ["amber/ff14SB.xml"]
    system_generator = SystemGenerator(forcefields=ffxmls, small_molecule_forcefield="openff_unconstrained-2.0.0", molecules=None,
                                       cache=None, removeCMMotion=False)
    # Note: openmmforcefields can parameterize small molecules via OpenFF if you pass the molecule; however,
    # for simplicity here we will rely on SystemGenerator to do protein + ligand if we pass ligand as extra molecules.
    # Convert modeller to a PDB string and re-load with Modeller to ensure box size
    pdbfile = app.PDBFile.writeFile(modeller.topology, modeller.positions, open("temp_model.pdb", "w"))
    pdbfile = PDBFile("temp_model.pdb")
    modeller2 = Modeller(pdbfile.topology, pdbfile.positions)
    modeller2.addSolvent(system_generator.forcefield, model='tip3p', padding=box_padding*unit.angstrom)
    # We need to combine ligand parameters into the created system. To keep this cell practical we will create the system via system_generator.create_system
    system = system_generator.create_system(modeller2.topology, nonbondedMethod=app.PME, constraints=app.HBonds)
    # Note: This is a simplified route. If the small-molecule params were created externally (ligand_system), you would need to merge the ligand_system into 'system'. That merging is non-trivial and requires matching indices.
    return system, modeller2.topology, modeller2.positions


# %%
# Simulation runner

def run_simulation(pdb_name: str, pdb_path: str, outdir: str, run_config: dict):
    """High-level wrapper: loads/fixes pdb, extracts ligand, parameterizes, builds system, runs minimize/equil/production, saves outputs.
    Returns paths to saved topology and trajectory.
    """
    outdir = make_output_dirs(outdir, pdb_name)
    print(f"Processing {pdb_name} -> {outdir}")
    # Load and fix
    pdbfile, modeller = load_and_fix_pdb(pdb_path, keep_chains=("A", "B"), ph=run_config.get("pH", PH))
    # Save cleaned topology
    cleaned_pdb_path = outdir / f"{pdb_name}_cleaned.pdb"
    with open(cleaned_pdb_path, "w") as fh:
        PDBFile.writeFile(modeller.topology, modeller.positions, fh)

    # Extract ligand and parameterize
    ligand_sdf = outdir / "ligand.sdf"
    off_mol = extract_ligand_to_sdf(modeller, ligand_chain="A", out_sdf_path=str(ligand_sdf))
    ligand_system, ligand_top = parameterize_ligand_openff(off_mol)

    # Build combined system (simplified)
    system, topology, positions = build_system(modeller, ligand_system, ligand_top, box_padding=run_config.get("box_padding", BOX_PADDING))

    # Set up integrator & platform
    integrator = mm.LangevinIntegrator(run_config.get("temperature", TEMPERATURE)*unit.kelvin,
                                       1.0/unit.picoseconds,
                                       run_config.get("dt", DT)*unit.picoseconds)
    if CONSTRAIN_HBONDS:
        integrator.setConstraintTolerance(1e-6)

    platform = mm.Platform.getPlatformByName(PLATFORM_NAME)
    properties = PLATFORM_PROPERTIES
    simulation = app.Simulation(topology, system, integrator, platform, properties)
    simulation.context.setPositions(positions)

    # Minimize
    print("Minimizing...")
    simulation.minimizeEnergy(maxIterations=run_config.get("minimize_steps", N_STEPS_MINIMIZE))

    # Equilibration NVT
    print("Equilibrating NVT...")
    simulation.context.setVelocitiesToTemperature(run_config.get("temperature", TEMPERATURE)*unit.kelvin)
    simulation.reporters.append(app.StateDataReporter(str(outdir / "log.txt"), 1000, step=True, potentialEnergy=True, temperature=True, progress=True))
    simulation.step(run_config.get("n_steps_eq_nvt", N_STEPS_EQ_NVT))

    # Equilibration NPT
    print("Equilibrating NPT...")
    simulation.reporters.append(app.StateDataReporter(str(outdir / "log_npt.txt"), 1000, step=True, potentialEnergy=True, temperature=True, volume=True))
    # Add barostat
    system.addForce(mm.MonteCarloBarostat(PRESSURE*unit.atmospheres, TEMPERATURE*unit.kelvin, 25))
    simulation = app.Simulation(topology, system, integrator, platform, properties)
    simulation.context.setPositions(positions)
    simulation.context.setVelocitiesToTemperature(run_config.get("temperature", TEMPERATURE)*unit.kelvin)
    simulation.step(run_config.get("n_steps_eq_npt", N_STEPS_EQ_NPT))

    # Production
    print("Starting production run...")
    traj_path = outdir / f"{pdb_name}_prod.dcd"
    # Save DCD reporter
    simulation.reporters.append(app.DCDReporter(str(traj_path), int(run_config.get("save_every", SAVE_EVERY_N_STEPS))))
    # Save checkpoint occasionally
    simulation.reporters.append(app.CheckpointReporter(str(outdir / "checkpoint.chk"), 50000))
    simulation.step(run_config.get("n_steps_prod", N_STEPS_PRODUCTION))

    # Save final positions
    final_pdb = outdir / f"{pdb_name}_final.pdb"
    positions = simulation.context.getState(getPositions=True).getPositions()
    with open(final_pdb, "w") as fh:
        PDBFile.writeFile(simulation.topology, positions, fh)

    print(f"Done: saved traj to {traj_path} and final pdb to {final_pdb}")
    return cleaned_pdb_path, traj_path


# %%
# Analysis utilities using MDAnalysis and MDTraj

def analyze_trajectory(pdb_path: str, traj_path: str, outdir: str):
    """Compute RMSD (protein backbone), ligand RMSD, ligand-protein distances, and H-bond occupancies.
    Saves plots and CSVs to outdir/analysis
    """
    u = mda.Universe(pdb_path, traj_path)
    protein = u.select_atoms("protein and backbone")
    ligand = u.select_atoms("chainid A and not name H*")

    # RMSD of protein backbone
    R = rms.RMSD(u, select="protein and backbone", ref_frame=0)
    R.run()
    rmsd_times = R.rmsd[:,1]  # time (frames) - depends on MDAnalysis version
    rmsd_vals = R.rmsd[:,2]
    df = pd.DataFrame({"frame": np.arange(len(rmsd_vals)), "rmsd_backbone_nm": rmsd_vals})
    df.to_csv(Path(outdir)/"analysis"/"rmsd_backbone.csv", index=False)

    # Ligand heavy-atom RMSD vs first frame
    from MDAnalysis.analysis.rms import rmsd
    ligand_coords0 = ligand.positions.copy()
    ligand_rms = []
    for ts in u.trajectory:
        coords = ligand.positions
        ligand_rms.append(rmsd(coords, ligand_coords0, superposition=False))
    df2 = pd.DataFrame({"frame": np.arange(len(ligand_rms)), "ligand_rmsd_nm": ligand_rms})
    df2.to_csv(Path(outdir)/"analysis"/"ligand_rmsd.csv", index=False)

    # Ligand center-of-mass distance to protein binding-site COM (protein whole)
    prot_com = protein.center_of_mass()
    ligand_coms = []
    for ts in u.trajectory:
        ligand_coms.append(ligand.center_of_mass())
    ligand_coms = np.array(ligand_coms)
    dists = np.linalg.norm(ligand_coms - prot_com, axis=1)
    pd.DataFrame({"frame": np.arange(len(dists)), "ligand_protein_com_dist_A": dists}).to_csv(Path(outdir)/"analysis"/"ligand_prot_com_dist.csv", index=False)

    # Hydrogen bonds (ligand as acceptor/donor to protein)
    hb_analysis = hbonds.HydrogenBondAnalysis(u, selection1="protein", selection2="chainid A", distance=3.5, angle=150.0)
    hb_analysis.run()
    # hb_analysis.results contains counts per frame; create a simple occupancy table
    # For a quick summary: total hbonds detected across trajectory and average per frame
    total_hbonds = np.sum([len(x) for x in hb_analysis.hbonds])
    avg_per_frame = total_hbonds / len(u.trajectory)
    pd.DataFrame({"total_hbonds": [total_hbonds], "avg_per_frame": [avg_per_frame]}).to_csv(Path(outdir)/"analysis"/"hb_summary.csv", index=False)

    print("Analysis saved to", Path(outdir)/"analysis")


# %%
# Snapshot exporter for MM/PBSA via AmberTools (if you have MMPBSA.py installed)

def export_snapshots_for_mmpbsa(pdb_path: str, traj_path: str, outdir: str, n_snapshots=50):
    """Extract evenly spaced snapshots and write AMBER-style topology/coordinates (.prmtop/.inpcrd or PDB) to run MMPBSA.py
    This function writes PDB snapshots; the user must create prmtop/inpcrd or AMBER-style topologies separately (e.g., via tleap/parmchk2)
    """
    traj = md.load(str(traj_path), top=str(pdb_path))
    n_frames = traj.n_frames
    indices = np.linspace(0, n_frames-1, min(n_snapshots, n_frames)).astype(int)
    out_snap_dir = Path(outdir)/"snapshots"
    for i, idx in enumerate(indices):
        frame = traj[idx]
        frame.save_pdb(str(out_snap_dir / f"snapshot_{i:04d}.pdb"))
    print(f"Exported {len(indices)} snapshots to {out_snap_dir}. Use AmberTools to build prmtop/inpcrd for MMPBSA.py")


# %%
# Batch runner: loop over all PDBs in INPUT_DIR

def batch_process(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR, run_config=None):
    run_config = run_config or {}
    input_dir = Path(input_dir)
    files = sorted([p for p in input_dir.iterdir() if p.suffix.lower() in ('.pdb',)])
    summary_rows = []
    for p in files:
        pdb_name = p.stem
        try:
            cleaned_pdb, traj = run_simulation(pdb_name, str(p), output_dir, run_config)
            analyze_trajectory(str(cleaned_pdb), str(traj), Path(output_dir)/pdb_name)
            export_snapshots_for_mmpbsa(str(cleaned_pdb), str(traj), Path(output_dir)/pdb_name)
            summary_rows.append({"pdb": pdb_name, "status": "ok"})
        except Exception as e:
            print(f"Error processing {pdb_name}: {e}")
            summary_rows.append({"pdb": pdb_name, "status": "error", "error": str(e)})
    pd.DataFrame(summary_rows).to_csv(Path(output_dir)/"summary.csv", index=False)
    print("Batch finished. Summary saved.")


# %%
# If you want to run from the notebook: set a smaller production length for testing
if __name__ == '__main__':
    # Example quick test config (short)
    test_config = {
        'pH': 7.0,
        'box_padding': 8.0,
        'minimize_steps': 2000,
        'n_steps_eq_nvt': 25000,
        'n_steps_eq_npt': 25000,
        'n_steps_prod': 250000,
        'save_every': 2500,
        'temperature': 300.0,
        'dt': 0.002,
    }
    # Uncomment to run batch automatically (be sure INPUT_DIR is set)
    # batch_process(INPUT_DIR, OUTPUT_DIR, run_config=test_config)

# %%
# Additional notes
# - The merging of OpenFF ligand System into the protein system is an advanced step. For a robust pipeline you may want to
#   let openmmforcefields/SystemGenerator parameterize the small molecule directly, or use `openmmforcefields` small molecule path.
# - For MM/PBSA with AmberTools, convert the OpenMM system to AMBER prmtop/inpcrd (or re-parameterize with AmberTools/antechamber) then run MMPBSA.py.
# - For higher-quality estimates consider running 3 replicates of production with different random seeds and checking convergence.
# - For alchemical free energy calculations (FEP) consider Perses or Yank; they require specialized setup.

